In [1]:
# Stage 1, HW 3
import numpy as np
import pandas as pd
import networkx as nx
import slider_logs as sl

table = pd.read_csv("runs_table.csv")
runs = sl.parse_all()   # same order as the rows of runs_table.csv
assert all(run["file"] == f and run["wave"] == w for run, f, w in zip(runs, table["file"], table["wave"]))
print(len(table), "runs")

/home/aalok/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


718 runs


In [2]:
# fraught_mask and the two ways to get % fraught (same as HW 2)
def fraught_mask(run):
    A = run["A"]                         # 7x7, no self-loops
    s = sl.sign_correct(run)             # s = weights * X, shape T x 7
    no_zeros = np.all(s != 0, axis=1)
    mixed = ~(np.all(s > 0, axis=1) | np.all(s < 0, axis=1))
    delta = np.sign(s) * (s @ A)         # row k of s @ A is A @ s_k (A is symmetric)
    no_switch = np.all(delta >= 0, axis=1)
    return no_zeros & mixed & no_switch

# method 1, forward hold: each row lasts until the next row
def fraught_percent_hold(i):
    run = runs[i]
    t = run["t"]
    if table["success"][i]:
        T_end = t[-1]                    # the consensus row
    elif table["session"][i] == "lab":
        T_end = max(300, t[-1])          # lab time limit
    else:
        T_end = max(180, t[-1])          # cloud time limit
    dt = np.diff(np.append(t, T_end))    # how long each row lasts
    return 100 * np.sum(dt * fraught_mask(run)) / T_end

# method 2, interpolation (authors' Matlab code): join the rows with straight lines,
# take 1001 evenly spaced times from the first row to the last row
def fraught_percent_interp(i, n=1000):
    run = runs[i]
    t = run["t"]
    s = sl.sign_correct(run)
    if len(t) < 2:
        return 0.0
    grid = np.linspace(t[0], t[-1], n + 1)
    s_grid = np.column_stack([np.interp(grid, t, s[:, j]) for j in range(7)])
    # weights = 1 because s_grid is already sign-corrected
    mask = fraught_mask({"A": run["A"], "weights": np.ones(7), "X": s_grid})
    return 100 * np.mean(mask)

In [3]:
# 3.1 % fraught for every run in EL, CL and CC (runs with no end marker are left out)
keep = table["dataset"].isin(["EL", "CL", "CC"]) & (table["end"] != "incomplete")

res = table.loc[keep, ["file", "dataset", "wave", "topology", "success", "duration", "bot"]].copy()
res["fraught % (hold)"] = [fraught_percent_hold(i) for i in res.index]
res["fraught % (interp)"] = [fraught_percent_interp(i) for i in res.index]

print(len(res), "runs")
res.to_csv("runs_fraught.csv", index=False)
res.round(2)

542 runs


,file,dataset,wave,topology,success,duration,bot,fraught % (hold),fraught % (interp)
0,log_Fri Sep 16 161000 EDT 2016.txt,CL,1,Ring,False,300.00,False,11.67,15.98
1,log_Fri Sep 16 161000 EDT 2016.txt,CL,2,Hierarchy,True,59.00,False,18.64,21.18
2,log_Fri Sep 16 161000 EDT 2016.txt,CL,3,Full,True,90.00,False,0.00,0.00
3,log_Fri Sep 16 161000 EDT 2016.txt,CL,4,Hierarchy,True,80.00,False,11.25,11.89
4,log_Fri Sep 16 161000 EDT 2016.txt,CL,5,Full,True,21.00,False,0.00,0.00
...,...,...,...,...,...,...,...,...,...
713,Log_2018-05-31_183920.txt,CC,5,Star,True,11.34,False,0.00,0.00
714,Log_2018-05-31_183920.txt,CC,6,Full,True,105.95,False,0.00,0.00
715,Log_2018-05-31_183920.txt,CC,7,Hierarchy,True,18.81,False,0.00,0.00
716,Log_2018-05-31_183920.txt,CC,8,Full,True,17.97,False,0.00,0.00


In [4]:
# 3.2 fraught volume: % of the cube [-1, 1]^7 that is fraught (Monte Carlo, as in Stage 0)
hierarchy = [(1, 2), (1, 3), (2, 4), (2, 5), (3, 6), (3, 7)]
edges = {
    "Full":         [(i, j) for i in range(1, 8) for j in range(i + 1, 8)],
    "Star":         [(1, 2), (1, 3), (1, 4), (1, 5), (1, 6), (1, 7)],
    "Hierarchy":    hierarchy,
    "CT-Hierarchy": hierarchy + [(4, 5), (6, 7)],
    "Ring":         [(1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 1)],
}

rng = np.random.default_rng(0)
S = rng.uniform(-1, 1, size=(10**6, 7))

volume = {}
for name, e in edges.items():
    A = nx.to_numpy_array(nx.Graph(e), nodelist=range(1, 8))
    volume[name] = 100 * np.mean(fraught_mask({"A": A, "weights": np.ones(7), "X": S}))
    print(f"{name:13s} fraught volume = {volume[name]:.3f} %")

Full          fraught volume = 0.000 %


Star          fraught volume = 0.000 %


Hierarchy     fraught volume = 1.291 %


CT-Hierarchy  fraught volume = 1.364 %
Ring          fraught volume = 1.220 %


In [5]:
# 3.2 table by dataset x topology
res["> 10% (hold)"] = res["fraught % (hold)"] > 10
res["> 10% (interp)"] = res["fraught % (interp)"] > 10

g = res.groupby(["dataset", "topology"])
summary = pd.DataFrame({
    "n": g.size(),
    "success rate %": 100 * g["success"].mean(),
    "mean % (hold)": g["fraught % (hold)"].mean(),
    "mean % (interp)": g["fraught % (interp)"].mean(),
    "# > 10% (hold)": g["> 10% (hold)"].sum(),
    "# > 10% (interp)": g["> 10% (interp)"].sum(),
})

# attraction ratio = mean % fraught / fraught volume %   (NaN where the volume is 0)
summary["volume %"] = [volume[topology] for dataset, topology in summary.index]
summary["ratio (hold)"] = summary["mean % (hold)"] / summary["volume %"]
summary["ratio (interp)"] = summary["mean % (interp)"] / summary["volume %"]
summary.round(2)

n  success rate %  mean % (hold)  mean % (interp)  \
dataset topology                                                           
CC      Full          66           96.97           0.00             0.00   
        Hierarchy     70           94.29          13.70            13.75   
        Star          68           98.53           0.00             0.00   
CL      Full          63           82.54           0.00             0.00   
        Hierarchy     63           92.06          11.58            12.23   
        Ring          59           89.83          15.76            15.86   
EL      CT-Hierarchy  51           82.35          17.62            17.97   
        Hierarchy     51           94.12          12.12            12.05   
        Ring          51           88.24          21.56            20.97   

                      # > 10% (hold)  # > 10% (interp)  volume %  \
dataset topology                                                   
CC      Full                       0                 0      0.00   
        Hierarchy                 23                24      1.29   
        Star                       0                 0      0.00   
CL      Full                       0                 0      0.00   
        Hierarchy                 18                19      1.29   
        Ring                      29                28      1.22   
EL      CT-Hierarchy              18                18      1.36   
        Hierarchy                 14                15      1.29   
        Ring                      27                27      1.22   

                      ratio (hold)  ratio (interp)  
dataset topology                                    
CC      Full                   NaN             NaN  
        Hierarchy            10.61           10.65  
        Star                   NaN             NaN  
CL      Full                   NaN             NaN  
        Hierarchy             8.97            9.48  
        Ring                 12.91           13.00  
EL      CT-Hierarchy         12.92           13.17  
        Hierarchy             9.39            9.33  
        Ring                 17.67           17.19

In [6]:
# 3.3 H1 test: paired, within-group comparison (group = one log file)
from scipy.stats import wilcoxon

# rule: drop the 1-row aborted rounds (marked failed after a single row, so they contain no play).
# They stay in 3.1 and 3.2, which are compared with the paper's numbers.
aborted = (table["T"] == 1) & (table["end"] == "failed")
play = res[~aborted[res.index]]
print("rounds dropped:", len(res) - len(play))

def paired_test(datasets, top_a, top_b, column):
    d = play[play["dataset"].isin(datasets)]
    means = d.groupby(["file", "topology"])[column].mean().unstack()   # one row per group
    diff = (means[top_a] - means[top_b]).dropna().values               # d_g (a group with no round of one topology is dropped)

    # bootstrap 95% CI of the mean of d_g: resample the groups 10,000 times
    rng = np.random.default_rng(0)
    boot = rng.choice(diff, size=(10000, len(diff))).mean(axis=1)
    low, high = np.percentile(boot, [2.5, 97.5])

    p = wilcoxon(diff).pvalue                                          # signed-rank test, two-sided
    return [len(diff), (diff > 0).sum(), diff.mean(), low, high, p]

tests = [
    ("CL",      ["CL"],       "Ring"),
    ("EL",      ["EL"],       "Ring"),
    ("CL + EL", ["CL", "EL"], "Ring"),
    ("EL",      ["EL"],       "CT-Hierarchy"),
]

rows = []
for label, datasets, top_a in tests:
    for method in ["hold", "interp"]:
        result = paired_test(datasets, top_a, "Hierarchy", "fraught % (" + method + ")")
        rows.append([label, top_a + " - Hierarchy", method] + result)

h1 = pd.DataFrame(rows, columns=["dataset", "comparison", "method", "groups", "groups d > 0", "mean d", "CI low", "CI high", "Wilcoxon p"])
h1.round(3)

rounds dropped: 3


,dataset,comparison,method,groups,groups d > 0,mean d,CI low,CI high,Wilcoxon p
0,CL,Ring - Hierarchy,hold,20,14,4.712,-3.318,12.428,0.231
1,CL,Ring - Hierarchy,interp,20,12,4.160,-4.123,12.148,0.349
2,EL,Ring - Hierarchy,hold,17,12,9.432,-0.671,18.338,0.027
3,EL,Ring - Hierarchy,interp,17,13,8.922,-1.173,17.820,0.031
4,CL + EL,Ring - Hierarchy,hold,37,26,6.880,0.445,12.866,0.017
5,CL + EL,Ring - Hierarchy,interp,37,25,6.348,-0.182,12.421,0.032
6,EL,CT-Hierarchy - Hierarchy,hold,17,13,5.500,-5.603,15.004,0.080
7,EL,CT-Hierarchy - Hierarchy,interp,17,13,5.922,-5.350,15.461,0.064
